In [16]:
import time
import requests
import pandas as pd

API = "https://sa.wikisource.org/w/api.php"
CONTACT = "tyler.g.neill@gmail.com"  # <-- change this

In [13]:
UA = f"sa-wikisource-uncat-sweep/0.1 ({CONTACT}) requests"

S = requests.Session()
S.headers.update({
    "User-Agent": UA,
    "Accept": "application/json",
})

def api_get(params, timeout=(10, 60), retries=6, backoff_s=2):
    last = None
    for attempt in range(retries):
        r = S.get(API, params=params, timeout=timeout)
        if r.status_code == 200:
            return r.json()
        last = RuntimeError(f"HTTP {r.status_code}: {r.text[:300]!r}")
        time.sleep(backoff_s * (2 ** attempt))
    raise last

def fetch_uncategorized_titles(qplimit=500, sleep_s=0.2):
    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "list": "querypage",
        "qppage": "Uncategorizedpages",
        "qplimit": str(qplimit),
        "maxlag": "5",
    }

    rows = []
    while True:
        data = api_get(params)
        results = data.get("query", {}).get("querypage", {}).get("results", [])
        rows.extend(results)

        cont = data.get("continue")
        if not cont:
            break
        params.update(cont)
        time.sleep(sleep_s)

    df = pd.DataFrame(rows)
    # columns usually include: ns, title, value (value = some score used by querypage)
    return df[["ns", "title"]].drop_duplicates().sort_values("title").reset_index(drop=True)

df_uncat_titles = fetch_uncategorized_titles()
df_uncat_titles

,ns,title
0,0,Bible
1,0,Celebrating Sanskrit
2,0,Chandrapeedacharitam
3,0,Chézy - La Reconnaissance de Sacountala
4,0,Chézy-2
...,...,...
4995,0,निघण्टुशास्त्रम्/द्वितीयोध्यायः
4996,0,निघण्टुशास्त्रम्/प्रथमोध्यायः
4997,0,निद्रा मन्त्रम्
4998,0,निरुक्तशास्त्रम्/अष्टमोध्यायः


In [52]:
import time
import sys

def iterate_allpages(batch_size=50, sleep_s=0.1):
    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "list": "allpages",
        "apnamespace": "0",
        "aplimit": str(batch_size),
        "maxlag": "5",
    }

    total = 0

    while True:
        data = S.get(API, params=params, timeout=(10, 60)).json()
        pages = data["query"]["allpages"]

        total += len(pages)
        sys.stdout.write(f"{total} … ")
        sys.stdout.flush()

        yield pages

        cont = data.get("continue")
        if not cont:
            break

        params.update(cont)
        time.sleep(sleep_s)

    print()  # final newline


In [60]:
def find_uncategorized_titles(titles, sleep_s=0.1):
    payload = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "titles": "|".join(titles),
        "prop": "categories",
        "cllimit": "1",      # existence check only
        "redirects": "1",
        "maxlag": "5",
    }

    r = S.post(API, data=payload, timeout=(10, 60))
    data = r.json()

    uncat = []
    for p in data.get("query", {}).get("pages", []):
        if p.get("missing"):
            continue
        if not p.get("categories"):
            uncat.append(p["title"])

    time.sleep(sleep_s)
    return uncat

In [62]:
uncategorized = []

for i, batch in enumerate(iterate_allpages(batch_size=50), start=1):
    titles = [p["title"] for p in batch]

    # progress indicator
    if i % 20 == 0:
        print(f"\nuncategorized so far: {len(uncategorized)}")
        if titles:
            print(f"\n@ batch {i}: {titles[0]}")

    uncat = find_uncategorized_titles(titles)
    uncategorized.extend(uncat)

50 … 100 … 150 … 200 … 250 … 300 … 350 … 400 … 450 … 500 … 550 … 600 … 650 … 700 … 750 … 800 … 850 … 900 … 950 … 1000 … 
uncategorized so far: 911

@ batch 20: अथर्ववेदः/काण्डं २/सूक्तम् ०९
1050 … 1100 … 1150 … 1200 … 1250 … 1300 … 1350 … 1400 … 1450 … 1500 … 1550 … 1600 … 1650 … 1700 … 1750 … 1800 … 1850 … 1900 … 1950 … 2000 … 
uncategorized so far: 1891

@ batch 40: अयोध्याकाण्डे षट्सप्ततितमः सर्गः ॥२-७६॥
2050 … 2100 … 2150 … 2200 … 2250 … 2300 … 2350 … 2400 … 2450 … 2500 … 2550 … 2600 … 2650 … 2700 … 2750 … 2800 … 2850 … 2900 … 2950 … 3000 … 
uncategorized so far: 2854

@ batch 60: उद्धवगीता ६
3050 … 3100 … 3150 … 3200 … 3250 … 3300 … 3350 … 3400 … 3450 … 3500 … 3550 … 3600 … 3650 … 3700 … 3750 … 3800 … 3850 … 3900 … 3950 … 4000 … 
uncategorized so far: 3849

@ batch 80: ऋग्वेद: सूक्तं ८.९८
4050 … 4100 … 4150 … 4200 … 4250 … 4300 … 4350 … 4400 … 4450 … 4500 … 4550 … 4600 … 4650 … 4700 … 4750 … 4800 … 4850 … 4900 … 4950 … 5000 … 
uncategorized so far: 4846

@ batch 100: ऋग्वेदः सूक्त

In [63]:
len(uncategorized)

31084